# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmairAsim180/1_FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring (feeding a ranking). My lane needs to produce a continuous priority score per page so a reviewer can sort their queue and work top-down instead of checking every page. Under the hood I'd likely train this as a binary classifier (declining vs. not) and use the predicted probability as the score — but the actual deliverable is an ordered list, not a hard yes/no label. That's why I'm calling it scoring rather than pure classification: what matters is relative order at the top of the list, not where exactly the decision boundary sits.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

df['trend_direction'].value_counts(normalize=True)


Loaded: (30000, 44)


,proportion
trend_direction,
down,0.542067
stable,0.198733
up,0.146267
new,0.074533
flat,0.038400


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/proxy: trend_direction == "down" — is this page currently declining in traffic. This is a proxy, not the true thing I care about ("does this page need a refresh"). The label comes from an observed signal computed over a trailing window, not a manual editorial judgment call — so it's outcome-based, not rule-defined. It's a reasonable stand-in because declining + high-impression pages are exactly what reviewers want caught early, but it will miss pages that haven't started declining yet and might benefit from a preemptive refresh — a limitation worth naming.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['trend_direction'].value_counts()


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@50. Reviewers have a fixed queue each cycle — say 50 pages — not the bandwidth to check all 30,000. Overall accuracy doesn't reflect that constraint; Precision@50 does: of the top 50 pages the model flags, what fraction are truly declining? My ML-02 baseline already gave me real numbers here — a fixed rule hits 0.240 Precision@50, a random forest hits 0.740 — so that's the number this whole lane is built around protecting.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
capacity = 50
print(f"Weekly review capacity covers {capacity/len(df)*100:.2f}% of all pages — precision at the top matters more than coverage.")


Weekly review capacity covers 0.17% of all pages — precision at the top matters more than coverage.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one content page, described by its trailing 90-day signals.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Columns:", list(df.columns))
print("Shape:", df.shape)
df.head()

Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

I already have direct evidence from ML-02: a hand-written rule (days_since_last_update >= 180 AND impressions_90d >= 500) hits only 0.240 Precision@50, while a random forest trained on the same signals hits 0.740 — roughly a 3x lift. A single if-statement can only threshold one or two signals at a time; it can't catch interactions like "this page is only mildly stale but impressions are dropping fast." That kind of combined, nonlinear signal is exactly what tree-based models pick up, and the gap between rule and model on my own data proves it isn't theoretical.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
comparison = pd.DataFrame({
    "method": ["hand-written rule", "random forest"],
    "precision_at_50": [0.240, 0.740]
})
comparison

,method,precision_at_50
0,hand-written rule,0.24
1,random forest,0.74


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.